# R5.6 — Is the gain from better labels or from the downstream model?

Reviewer 5, comment 6: evaluate the cleaned datasets with several classifiers under one
identical protocol, against the noisy data and an oracle-clean dataset.

Training sets, all from the same published noisy labels:
`noisy` (as observed) · `ours` (DetectAndRelabel at the oracle thresholds `td`, `tr`) ·
`oracle_filtered` (every truly noisy sample removed: separates *fewer samples* from *better
labels*) · `oracle_clean` (all ground-truth labels: the ceiling).

Classifiers: PreAct-ResNet34 (the paper's), ResNet-18, VGG-16-BN, MobileNetV2 (32×32 stems).
Protocol = the paper's `FinalModelTester` settings for the dataset (Adam, 5-epoch linear warm-up
then constant rate, label smoothing 0.1, stratified validation split, early stopping on
validation accuracy, best weights, clean test set), applied unchanged to every classifier and
training set.

The `ours` × `preact_resnet34` cells follow the paper's Table 3 protocol, so with
`preds_ref='main'` they are also the downstream numbers for the re-run CIFAR-10 predictions
(the paper's Table 3 still reports the round-3 run, `preds_ref='513d833'`).

**Budget (rough):** CIFAR-10 ≈ 0.4–0.8 h per job on a T4; 4 sets × 4 classifiers × 3 seeds = 48 jobs
≈ 30 GPU-h ≈ 15 h on T4×2. Jobs run seed by seed, so each commit leaves complete seeds.

**Kaggle settings:** Accelerator *GPU T4 x2* (P100 also works, one worker), Internet *On*
(phone-verified account), Persistence *Files only*. Run with **Save Version -> Save & Run All
(Commit)** so it runs in the background for up to 12 h.

**Resuming:** jobs are saved as they finish. If the session ends with jobs left, open the
notebook, *Add Input -> Your Work -> this notebook* (the previous version's output), and
commit again: finished jobs are restored and skipped. Repeat until the launch cell prints
that nothing is left; the analysis cell then reports the final tables.

**Outputs** (in the version's Output tab): `r56_out/` raw per-job results, `results/` CSV
tables, `logs/` per-GPU training logs.

## 1. Configuration

In [ ]:
import json, os
# Oracle thresholds (td, tr) per run. 'main' = the CIFAR-10 re-run (best noise-F1 on the paper's
# 6..10 grid with tr=10 -- confirm before use); '513d833' = the round-3 paper run (its Table 2).
# Fashion-MNIST predictions are the same in both.
THRESHOLDS = {
    ('main', 'cifar10', 20): (10, 10), ('main', 'cifar10', 30): (8, 10), ('main', 'cifar10', 40): (8, 10),
    ('513d833', 'cifar10', 20): (8, 10), ('513d833', 'cifar10', 30): (7, 10), ('513d833', 'cifar10', 40): (6, 10),
    ('main', 'fashionmnist', 20): (9, 10), ('main', 'fashionmnist', 30): (9, 10), ('main', 'fashionmnist', 40): (7, 10),
}
CONFIG = dict(
    dataset='cifar10', noise=20, preds_ref='main', code_ref='main',
    train_sets=['noisy', 'ours', 'oracle_filtered', 'oracle_clean'],
    archs=['preact_resnet34', 'resnet18', 'vgg16_bn', 'mobilenet_v2'],
    seeds=[0, 1, 2], amp=True,
    session_hours=11.5, job_hours=0.9, loader_workers=2,
)
CONFIG.update(json.loads(os.environ.get('SND_CONFIG_OVERRIDES', '{}')))   # local smoke tests only
CONFIG['td'], CONFIG['tr'] = THRESHOLDS[(CONFIG['preds_ref'], CONFIG['dataset'], CONFIG['noise'])]
OUT_NAME, WORKER, PIP = 'r56_out', 'r56_worker.py', ['timm']
N_JOBS = len(CONFIG['train_sets']) * len(CONFIG['archs']) * len(CONFIG['seeds'])

## 2. Setup (clone the repo, import kaggle/src)

In [ ]:
import json, os, subprocess, sys, time
SESSION_START = time.time()
os.chdir('/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd())
subprocess.run('nvidia-smi -L', shell=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + PIP, check=False)
REPO = os.environ.get('SND_REPO') or '/tmp/snd/SiameseNoiseDetection'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/stormaref/SiameseNoiseDetection.git',
                    REPO], check=True)
if not os.environ.get('SND_REPO'):                   # never move a local working copy
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', CONFIG['code_ref']], check=True)
sys.path.insert(0, os.path.join(REPO, 'kaggle', 'src'))
import snd_kaggle as K
repo = K.setup_repo(CONFIG['code_ref'], REPO)
WORKER = os.path.join(REPO, 'kaggle', 'src', WORKER)
COMMIT = subprocess.run(['git', '-C', repo, 'log', '-1', '--format=%H %ad %s', '--date=iso'],
                        capture_output=True, text=True).stdout.strip()
print('code:', COMMIT)

## 3. Train (all GPUs, resumable)

In [ ]:
for train in (True, False):                     # download once, before the workers start
    K.base_dataset(CONFIG['dataset'], train)
if 'variants' in CONFIG:                          # R5.1: pretrained backbone weights, once
    K.prefetch_backbone(CONFIG['dataset'], CONFIG['noise'])
CONFIG['out_dir'] = K.restore_previous_outputs(OUT_NAME)
CONFIG['session_start'] = SESSION_START
CONFIG['commit'] = COMMIT
with open(os.path.join(CONFIG['out_dir'], f'config_{int(SESSION_START)}.json'), 'w') as f:
    json.dump(CONFIG, f, indent=1)                  # provenance: config + commit per session
with open('config.json', 'w') as f:
    json.dump(CONFIG, f, indent=1)
codes = K.launch_workers(WORKER, 'config.json', poll=CONFIG.get('poll', 300))
print('worker exit codes:', codes)
done = len(K.JobQueue(CONFIG['out_dir']).summaries())
print(f'{done} of {N_JOBS} jobs finished' + ('' if done >= N_JOBS else
      ' -- commit a new version with this one attached as input to continue'))

## 4. Analysis

In [ ]:
import r56_analysis as A
_ = A.report(CONFIG['out_dir'], 'results')